# CLSS 2021 Replication
**Goulet Coulombe, Leroux, Stevanovic, Surprenant (2021)**  
*Macroeconomic data transformations matter*, IJF 37(4): 1338–1354

This notebook replicates and extends key results using the `macrocast` package.  
We work through setup, data loading, feature construction, and forecast evaluation step by step.

## 1. Data

Paper settings:
- **Vintage**: FRED-MD 2018-02 (February 2018 release)
- **Full sample**: 1959M01 – 2017M12
- **OOS start**: 1980M01
- **Estimation start**: 1960M01 (expanding window)

In [1]:
from macrocast.data import load_fred_md

# Load the paper vintage: FRED-MD 2018-02
md = load_fred_md(vintage="2018-02")

print(f"Shape      : {md.data.shape}")
print(f"Date range : {md.data.index[0]} ~ {md.data.index[-1]}")
print(f"Variables  : {md.data.shape[1]}")
print(f"\nFirst 5 columns : {list(md.data.columns[:5])}")
print(f"Last  5 columns : {list(md.data.columns[-5:])}")

Shape      : (709, 127)
Date range : 1959-01-01 00:00:00 ~ 2018-01-01 00:00:00
Variables  : 127

First 5 columns : ['RPI', 'W875RX1', 'DPCERA3M086SBEA', 'CMRMTSPLx', 'RETAILx']
Last  5 columns : ['MZMSL', 'DTCOLNVHFNM', 'DTCTHFNM', 'INVEST', 'VXOCLSx']


## 2. Target Variables

The paper forecasts 10 monthly macroeconomic indicators (Section 3).
We verify each series is present in the 2018-02 vintage and check its transformation code.

In [2]:
# Paper target variables: {paper label: FRED-MD series name}
# PPI: PPIFGS was renamed to WPSFD49207 ("PPI: Finished Goods") in the 2018-02 vintage
TARGETS = {
    "INDPRO":  "INDPRO",
    "EMP":     "PAYEMS",
    "UNRATE":  "UNRATE",
    "INCOME":  "RPI",
    "CONS":    "DPCERA3M086SBEA",
    "RETAIL":  "RETAILx",
    "HOUST":   "HOUST",
    "M2":      "M2REAL",
    "CPI":     "CPIAUCSL",
    "PPI":     "WPSFD49207",    # PPI: Finished Goods (same series, different code)
}

cols = md.data.columns.tolist()
vmeta = md.metadata.variables

print(f"{'Paper label':<10} {'FRED-MD series':<25} {'Found':>6} {'tcode':>6}")
print("-" * 52)
for label, series in TARGETS.items():
    found = series in cols
    tcode = vmeta[series].tcode if found else "--"
    print(f"{label:<10} {series:<25} {'YES' if found else 'NO':>6} {str(tcode):>6}")

Paper label FRED-MD series             Found  tcode
----------------------------------------------------
INDPRO     INDPRO                       YES      5
EMP        PAYEMS                       YES      5
UNRATE     UNRATE                       YES      2
INCOME     RPI                          YES      5
CONS       DPCERA3M086SBEA              YES      5
RETAIL     RETAILx                      YES      5
HOUST      HOUST                        YES      4
M2         M2REAL                       YES      5
CPI        CPIAUCSL                     YES      6
PPI        WPSFD49207                   YES      6


## 3. Data Preparation

Apply McCracken-Ng stationarity transformations (tcode 1–7) to the full panel.

**Why stationarity transformation?**
PCA (F, MAF) and MARX require stationary inputs to avoid spurious common-trend factors.
`append_raw_x` refers to stationary X columns. Only `append_levels=True` info sets
additionally use the original level form.

The trimmed panels (`panel`, `panel_levels`) are passed directly into `ForecastExperiment`,
which handles the expanding window split internally for each forecast origin.

In [3]:
import pandas as pd
import numpy as np

# Paper sample bounds (Appendix B)
SAMPLE_START = "1960-01-01"   # estimation window start (1960M01)
OOS_START    = "1980-01-01"   # OOS evaluation start   (1980M01)
OOS_END      = "2017-12-01"   # OOS evaluation end     (2017M12)

# Step 1: McCracken-Ng stationarity transformations (tcode 1-7)
md_stat   = md.transform()

# Step 2: Trim both panels to estimation window
#   panel:        stationary panel — main predictor matrix for PCA, MARX, raw X
#   panel_levels: level-form panel — used only for append_levels info sets
panel        = md_stat.trim(start=SAMPLE_START, end=OOS_END).data
panel_levels = md.trim(start=SAMPLE_START, end=OOS_END).data

print(f"Stationary panel : {panel.shape}  "
      f"({panel.index[0].date()} – {panel.index[-1].date()})")
print(f"Level panel      : {panel_levels.shape}")
print(f"NaN fraction     : {panel.isna().mean().mean():.3f}  "
      f"(imputed inside FeatureBuilder)")
print(f"\nOOS window passed to ForecastExperiment: {OOS_START[:7]} – {OOS_END[:7]}"
      f"  ({len(pd.date_range(OOS_START, OOS_END, freq='MS'))} months)")

Stationary panel : (696, 127)  (1960-01-01 – 2017-12-01)
Level panel      : (696, 127)
NaN fraction     : 0.010  (imputed inside FeatureBuilder)

OOS window passed to ForecastExperiment: 1980-01 – 2017-12  (456 months)


## 4. Paper Parameters and Information Sets

The paper (Appendix B) uses:
- **(P_y, k) = (12, 8)**: 12 AR lags of target, 8 PCA factors
- **P_MARX = 12**: MARX lag order
- **Horizons**: h ∈ {1, 3, 6, 9, 12, 24}

Table 1 defines 16 information sets (Z_t).  We load them directly from the
`CLSS2021` preset, which generates all 16 `FeatureSpec` objects with the correct
`factor_type` and `append_*` flag combinations.

In [4]:
from macrocast.pipeline.presets import CLSS2021

# Paper parameters (Appendix B)
P_Y    = 12    # AR lags of target (P_y)
K      = 8     # PCA factors
P_MARX = 12    # MARX lag order
HORIZONS = CLSS2021.HORIZONS  # [1, 3, 6, 9, 12, 24]

# All 16 Table 1 information sets via CLSS2021 preset
INFO_SETS = CLSS2021.info_sets(P_Y=P_Y, K=K, P_MARX=P_MARX)

# Quick inspection
print(f"Defined {len(INFO_SETS)} information sets  (horizons: {HORIZONS})\n")
print(f"{'Label':<16}  {'factor_type':<8}  {'flags'}")
print("-" * 60)
for label, spec in INFO_SETS.items():
    flags = ", ".join(f for f, v in [
        ("append_x_factors", spec.append_x_factors),
        ("append_marx",      spec.append_marx),
        ("append_raw_x",     spec.append_raw_x),
        ("append_levels",    spec.append_levels),
    ] if v) or "—"
    print(f"{label:<16}  {spec.factor_type:<8}  {flags}")

Defined 16 information sets  (horizons: [1, 3, 6, 9, 12, 24])

Label             factor_type  flags
------------------------------------------------------------
F                 X         —
F-X               X         append_raw_x
F-MARX            X         append_marx
F-MAF             MARX      append_x_factors
F-Level           X         append_levels
F-X-MARX          X         append_marx, append_raw_x
F-X-MAF           MARX      append_x_factors, append_raw_x
F-X-Level         X         append_raw_x, append_levels
F-X-MARX-Level    X         append_marx, append_raw_x, append_levels
X                 none      append_raw_x
MARX              none      append_marx
MAF               MARX      —
X-MARX            none      append_marx, append_raw_x
X-MAF             MARX      append_raw_x
X-Level           none      append_raw_x, append_levels
X-MARX-Level      none      append_marx, append_raw_x, append_levels
